# 05 — Refolding Evaluation

For each design from notebooks 01/03/04: run **ProteinMPNN → ESMFold** and check
self-consistency:

- Cα RMSD (design vs predicted) ≤ 2.0 Å
- mean ESMFold pLDDT ≥ 70

This is the canonical "designability" metric used in RFD and Chroma papers.

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import (RESULTS, DATA, kabsch_rmsd, load_ca_coords,
                   free_gpu, ensure_ligandmpnn, run_ligandmpnn)
import numpy as np, json, time, subprocess, os
from pathlib import Path

DESIGN_ROOTS = {
    ('chroma', 'uncond'):    DATA / 'chroma_uncond',
    ('rfd3',   'uncond'):    DATA / 'rfd3_uncond',
    ('chroma', 'sm_binder'): DATA / 'chroma_ligand',
    ('rfd3',   'sm_binder'): DATA / 'rfd3_ligand',
    ('chroma', 'enzyme'):    DATA / 'chroma_enzyme',
    ('rfd3',   'enzyme'):    DATA / 'rfd3_enzyme',
}
N_PER_GROUP = 6        # cap per (model, task) on T4
N_SEQS = 2             # MPNN samples per backbone

## Setup

In [ ]:
# ESMFold via fair-esm
try:
    import esm
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'fair-esm'], check=True)
    import esm

ensure_ligandmpnn()

In [ ]:
import torch

fold_model = esm.pretrained.esmfold_v1()
fold_model = fold_model.eval().cuda()
# fp16 fits on T4
try:
    fold_model.esm = fold_model.esm.half()
except Exception:
    pass
fold_model.set_chunk_size(64)

def fold(seq):
    with torch.no_grad():
        return fold_model.infer_pdb(seq)

def parse_ca_and_plddt(pdb_str):
    ca, plddts = [], []
    for line in pdb_str.split('\n'):
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            ca.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            plddts.append(float(line[60:66]))
    if not ca:
        return np.zeros((0, 3), np.float32), 0.0
    return np.array(ca, np.float32), float(np.mean(plddts))

## Refolding loop

In [ ]:
def parse_fasta(p):
    s, name = {}, None; cur = []
    for ln in open(p):
        if ln.startswith('>'):
            if name: s[name] = ''.join(cur)
            name, cur = ln[1:].strip(), []
        else:
            cur.append(ln.strip())
    if name: s[name] = ''.join(cur)
    return s

records = []
for (tag, task), root in DESIGN_ROOTS.items():
    if not root.exists():
        continue
    files = sorted(root.rglob('*.pdb'))
    files = [f for f in files if 'scaffold' not in f.name][:N_PER_GROUP]
    if not files:
        continue
    print(f'--- {tag} / {task}: {len(files)} designs ---')
    for pdb in files:
        seq_dir = pdb.parent / 'mpnn' / pdb.stem
        ok = run_ligandmpnn(pdb, seq_dir,
                             n_seqs=N_SEQS,
                             model_type='protein_mpnn',
                             checkpoint='proteinmpnn_v_48_020.pt')
        if not ok:
            continue
        seq_files = list((seq_dir / 'seqs').glob('*.fa'))
        if not seq_files:
            continue
        target_ca = load_ca_coords(pdb)
        for fa in seq_files:
            for sname, sseq in list(parse_fasta(fa).items())[:N_SEQS]:
                if not (30 <= len(sseq) <= 400):
                    continue
                try:
                    pred_pdb = fold(sseq)
                    pred_ca, plddt = parse_ca_and_plddt(pred_pdb)
                    L = min(len(target_ca), len(pred_ca))
                    if L < 30:
                        continue
                    rmsd = kabsch_rmsd(target_ca[:L], pred_ca[:L])
                    records.append({
                        'model': tag, 'task': task,
                        'design': pdb.stem, 'seq_id': sname,
                        'length': L, 'rmsd': rmsd, 'plddt': plddt,
                        'pass': bool(rmsd < 2.0 and plddt > 70),
                    })
                except torch.cuda.OutOfMemoryError:
                    free_gpu()
                except Exception as e:
                    print(f'  fold err: {type(e).__name__}: {str(e)[:80]}')
        free_gpu()

print(f'\nTotal refold records: {len(records)}')
(RESULTS / 'refold.json').write_text(json.dumps(records, indent=2))

## Aggregate & plot

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

if not records:
    print('No records — run notebooks 01/03/04 first.')
else:
    df = pd.DataFrame(records)
    agg = df.groupby(['model', 'task']).agg(
        pass_rate=('pass', 'mean'),
        n=('pass', 'count'),
        mean_rmsd=('rmsd', 'mean'),
        mean_plddt=('plddt', 'mean'),
    ).reset_index()
    print(agg)
    agg.to_csv(RESULTS / 'refold_summary.csv', index=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    tasks = sorted(df['task'].unique())
    w = 0.35
    for i, m in enumerate(['rfd3', 'chroma']):
        ys = []
        for t in tasks:
            sub = agg.query("model == @m and task == @t")
            ys.append(float(sub['pass_rate'].iloc[0]) if len(sub) else 0.0)
        ax.bar(np.arange(len(tasks)) + i*w, ys, w, label=m.upper())
    ax.set_xticks(np.arange(len(tasks)) + w/2)
    ax.set_xticklabels(tasks)
    ax.set_ylabel('Self-consistency pass rate')
    ax.set_ylim(0, 1)
    ax.set_title('Refolding pass rate (Cα RMSD < 2.0 Å & pLDDT > 70)')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RESULTS / 'fig_refold_pass.png', dpi=150)
    plt.show()